In [1]:
# fix relative imports
import os
cwd = os.path.normpath(os.getcwd())
cwd = cwd.split(os.sep)
find = cwd.index("fidelity-phase-tran")
newdir = f"{os.sep}".join(cwd[:find+1])
os.chdir(newdir)
%load_ext autoreload
%autoreload 2

# import necessay packages
import numpy as np
from ncon import ncon
from qs_mps.mps_class import MPS
import scipy.linalg as la
import pickle
import gzip

## import the mps data

In [170]:
filename = "results/data/ANNNI-8e2b8bce-1d27-41d0-a48b-fbce8d4ab8cc.pkl" # ANNNI 50, c1=-1, upside down, 64 x 64
# filename = "results/data/ANNNI-010201b6-5ab8-45af-9e50-507c9b609272.pkl" # if we truncate the bond dimension dmrg finds a product state, we can only see the disorder line
filename = "results/data/ANNNI-3f848197-11cd-4ce7-8e20-4166478a5ac9.pkl" # ANNNI 10, c1=-1, upside down, 10 x 10, all phase diagram


with gzip.open(filename, 'rb') as f:
    data = pickle.load(f)
params = data['params']
l, n = data['l'], data['n']
gstates = data['gstates']
stats = data['stats']

params_extent = np.concatenate([np.min(params, axis=0), np.max(params, axis=0)])
params_extent = tuple(params_extent[[0, 2, 1, 3]])

## define the functions for the improved rdms

We need the following ingredient:
 * sites we want to use for the rdms.There are two cases:
    1. Adjacent case (AC) - We can take only the interested sites and form a tensor because of Left and Right Canonical Form
    2. Non-Adjacent case (NAC) - We need to compute a middle environment that won't be in none fo the two canonical forms (LCF or RCF)
 * mps in Mixed Canonical Form (MCF). After an even number of half sweepings of the DMRG algorithm, the mps is in RCF. We need the states before the first site of our rdm to be in LCF  

If we have both ingredients, we can build efficient rdms where we ignore the left and right environments of our mps state. For the case *2.* of NAC we have an extra computation of sites in the middle but can be made efficiently taking into account the contraction cost.

In [171]:
def left_canonical_form(
        mps: list, stop: int, schmidt_tol: float=1e-15
    ):
        """
        right_svd

        This function transforms the states in self.sites in a canonical
        form using svd. We start from the first site and sweeping through
        site self.L we save the Gamma tensors on each site and the Schmidt values on the bonds

        schmidt_tol: float - tolerance used to cut the schmidt values after svd

        """
        new_mps = mps.copy()
        bonds = []
        s_init = np.array([1])
        psi = np.diag(s_init)
        d = mps[0].shape[1]
        bonds.append(s_init)

        for i in range(stop):
            new_site = ncon(
                [psi, new_mps[i]],
                [
                    [-1, 1],
                    [1, -2, -3],
                ],
            )
            new_site = new_site.reshape(new_site.shape[0] * d, new_site.shape[2])

            original_matrix = new_site
            scaled_matrix = original_matrix / np.max(np.abs(original_matrix))
            lambda_ = 1e-15
            regularized_matrix = scaled_matrix + lambda_ * np.eye(
                scaled_matrix.shape[0], scaled_matrix.shape[1]
            )
            u, s, v = la.svd(
                regularized_matrix,
                full_matrices=False,
            )

            bond_l = u.shape[0] // d
            u = u.reshape(bond_l, d, u.shape[1])
            condition = s >= schmidt_tol
            s_trunc = np.extract(condition, s)
            s = s_trunc / la.norm(s_trunc)
            u = u[:, :, : len(s)]
            v = v[: len(s), :]

            new_mps[i] = u
            bonds.append(s)
            psi = ncon(
                [np.diag(s), v],
                [
                    [-1, 1],
                    [1, -2],
                ],
            )

        return new_mps

def ac_rdm(mps, sites) -> np.array:
    kets = mps
    L = len(mps)
    
    d = kets[0].shape[1]
    k = len(sites)
    bras = [ket.conjugate() for ket in kets]
    a = np.array([1])
    env = ncon([a, a], [[-1], [-2]])
    up = [int(-elem) for elem in np.linspace(1, 0, 0)]
    down = [int(-elem) for elem in np.linspace(L + 1, 0, 0)]
    mid_up = [1]
    mid_down = [2]
    label_env = up + down + mid_up + mid_down
    # left env:
    env_l = env
    for i in range(sites[0] - 1):
        label_ket = [1, 3, -1]
        label_bra = [2, 3, -2]
        env_l = ncon([env_l, kets[i], bras[i]], [label_env, label_ket, label_bra])

    # # check that the env_l is indeed an identity matrix
    # env_l_shape = env_l.shape
    # env_l_flat = env_l.flatten()
    # env_l_trunc = []
    # for val in env_l_flat:
    #     if np.abs(val) < 1e-15:
    #         env_l_trunc.append(0)
    #     else:
    #         env_l_trunc.append(val)
    # env_l = np.array(env_l_trunc).reshape(env_l_shape)
    # print(env_l.shape, mps[sites[0]-1].shape[0])
    
    # right env:
    env_r = env
    for i in range(L - 1, sites[-1] - 1, -1):
        label_ket = [-1, 3, 1]
        label_bra = [-2, 3, 2]
        env_r = ncon([env_r, kets[i], bras[i]], [label_env, label_ket, label_bra])
    
    # # check that the env_r is indeed an identity matrix
    # env_r_shape = env_r.shape
    # env_r_flat = env_r.flatten()
    # env_r_trunc = []
    # for val in env_r_flat:
    #     if np.abs(val) < 1e-15:
    #         env_r_trunc.append(0)
    #     else:
    #         env_r_trunc.append(val)
    # env_r = np.array(env_r_trunc).reshape(env_r_shape)
    # print(env_r.shape, mps[sites[-1]-1].shape[-1])
    
    # central env
    # idx = 0
    for i in range(k):
        label_ket = [1, -1 - i, -k * 100]
        label_bra = [2, -k - 1 - i, -k * 100 - 1]
        env_l = ncon(
            [env_l, kets[sites[i] - 1], bras[sites[i] - 1]],
            [label_env, label_ket, label_bra],
        )
        up = [int(-elem) for elem in np.linspace(1, i + 1, i + 1)]
        down = [int(-elem) for elem in np.linspace(k + 1, k + 1 + i, i + 1)]
        label_env = up + down + mid_up + mid_down

    rdm = ncon([env_l, env_r], [label_env, [1, 2]]).reshape((d**k, d**k))

    return rdm

def improved_ac_rdm(mps, sites) -> np.array:
    kets = mps
    L = len(mps)
    
    d = kets[0].shape[1]
    k = len(sites)
    bras = [ket.conjugate() for ket in kets]
    a = np.array([1])
    env = ncon([a, a], [[-1], [-2]])
    up = [int(-elem) for elem in np.linspace(1, 0, 0)]
    down = [int(-elem) for elem in np.linspace(L + 1, 0, 0)]
    mid_up = [1]
    mid_down = [2]
    label_env = up + down + mid_up + mid_down
    # left env:
    env_l = np.identity(n=mps[sites[0]-1].shape[0])
    # right env:
    env_r = np.identity(n=mps[sites[-1]-1].shape[-1])
    
    # central env
    # idx = 0
    for i in range(k):
        label_ket = [1, -1 - i, -k * 100]
        label_bra = [2, -k - 1 - i, -k * 100 - 1]
        env_l = ncon(
            [env_l, kets[sites[i] - 1], bras[sites[i] - 1]],
            [label_env, label_ket, label_bra],
        )
        up = [int(-elem) for elem in np.linspace(1, i + 1, i + 1)]
        down = [int(-elem) for elem in np.linspace(k + 1, k + 1 + i, i + 1)]
        label_env = up + down + mid_up + mid_down

    rdm = ncon([env_l, env_r], [label_env, [1, 2]]).reshape((d**k, d**k))

    return rdm

In [35]:
# check the shapes
from qs_mps.utils import tensor_shapes
psi = gstates[0]()
tensor_shapes(psi, prnt=False)

[(1, 2, 2),
 (2, 2, 4),
 (4, 2, 8),
 (8, 2, 16),
 (16, 2, 32),
 (32, 2, 64),
 (64, 2, 64),
 (64, 2, 64),
 (64, 2, 64),
 (64, 2, 64),
 (64, 2, 64),
 (64, 2, 64),
 (64, 2, 64),
 (64, 2, 64),
 (64, 2, 64),
 (64, 2, 64),
 (64, 2, 64),
 (64, 2, 64),
 (64, 2, 64),
 (64, 2, 64),
 (64, 2, 64),
 (64, 2, 64),
 (64, 2, 64),
 (64, 2, 64),
 (64, 2, 64),
 (64, 2, 64),
 (64, 2, 64),
 (64, 2, 64),
 (64, 2, 64),
 (64, 2, 64),
 (64, 2, 64),
 (64, 2, 64),
 (64, 2, 64),
 (64, 2, 64),
 (64, 2, 64),
 (64, 2, 64),
 (64, 2, 64),
 (64, 2, 64),
 (64, 2, 64),
 (64, 2, 64),
 (64, 2, 64),
 (64, 2, 64),
 (64, 2, 64),
 (64, 2, 64),
 (64, 2, 32),
 (32, 2, 16),
 (16, 2, 8),
 (8, 2, 4),
 (4, 2, 2),
 (2, 2, 1)]

correct shape!

In [83]:
# check the canonical form of the mps
chain = MPS(L=len(psi), d=2, model="ANNNI")
chain.sites = psi.copy()
chain.check_canonical(site=-3)
psi_rcf = psi

the tensor at site 50 is in the correct RFC
the tensor at site 49 is in the correct RFC
the tensor at site 48 is in the correct RFC
the tensor at site 47 is in the correct RFC
the tensor at site 46 is in the correct RFC
the tensor at site 45 is in the correct RFC
the tensor at site 44 is in the correct RFC
the tensor at site 43 is in the correct RFC
the tensor at site 42 is in the correct RFC
the tensor at site 41 is in the correct RFC
the tensor at site 40 is in the correct RFC
the tensor at site 39 is in the correct RFC
the tensor at site 38 is in the correct RFC
the tensor at site 37 is in the correct RFC
the tensor at site 36 is in the correct RFC
the tensor at site 35 is in the correct RFC
the tensor at site 34 is in the correct RFC
the tensor at site 33 is in the correct RFC
the tensor at site 32 is in the correct RFC
the tensor at site 31 is in the correct RFC
the tensor at site 30 is in the correct RFC
the tensor at site 29 is in the correct RFC
the tensor at site 28 is in the 

correct mps canonical for after dmrg!

## Compare times

In [7]:
sites=[10]
for site in range(11,17):
    print(f"time for {len(sites)} site(s) rmd")
    %timeit ac_rdm(mps=psi_rcf, sites=sites)
    sites = sites + [site]

time for 1 site(s) rmd
12.7 ms ± 817 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)
time for 2 site(s) rmd
12.9 ms ± 713 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)
time for 3 site(s) rmd
14.8 ms ± 610 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)
time for 4 site(s) rmd
31.1 ms ± 2.12 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
time for 5 site(s) rmd
70.5 ms ± 2.78 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
time for 6 site(s) rmd
379 ms ± 7.45 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [8]:
sites=[10]
print(f"time for left canonical form")
%timeit left_canonical_form(mps=psi_rcf, stop=sites[0])
psi_mcf = left_canonical_form(mps=psi_rcf, stop=sites[0])
for site in range(11,17):
    print(f"time for {len(sites)} site(s) rmd")
    %timeit improved_ac_rdm(mps=psi_mcf, sites=sites)
    sites = sites + [site]

time for left canonical form
18.2 ms ± 951 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)
time for 1 site(s) rmd
1.28 ms ± 56.9 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
time for 2 site(s) rmd
2.34 ms ± 119 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)
time for 3 site(s) rmd
4.75 ms ± 250 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)
time for 4 site(s) rmd
17.6 ms ± 648 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)
time for 5 site(s) rmd
62.5 ms ± 3.61 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
time for 6 site(s) rmd
374 ms ± 11 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [6]:
def _compute_norm(mps, site, prnt=False):
    """
    _compute_norm

    This function computes the norm of our quantum state which is represented in mps.
    It takes the attributes .sites and .bonds of the class which gives us
    mps in canonical form (Vidal notation).

    site: int - sites in the chain
    ancilla: bool - if True we compute the norm of the ancilla_sites mps. By default False
    mixed: bool - if True we compute the braket between the ancilla_sites and the sites mps. By default False

    """
    a = np.array([1])

    array = mps.copy()

    ten = ncon([a, a],[[-1], [-2]])
    for i in range(site):
        
        ten = ncon(
            [ten, array[i], array[i].conjugate()],
            [[1, 2], [1, 3, -1], [2, 3, -2]],
        )
        if prnt:
            print(f"site {i+1}: ", ten.shape, array[i].shape)

    N = ncon([ten, a, a], [[1, 2], [1], [2]]).real

    return N

def generalized_k_rdm(mps, sites, prnt: bool=False) -> np.array:

    psi_mcf = left_canonical_form(mps=mps, stop=sites[0])
    norm = _compute_norm(psi_mcf, len(psi_mcf), prnt=False)
    psi_mcf[sites[0]] = psi_mcf[sites[0]]/(norm**(1/2))

    kets = mps
    bras = [ket.conjugate() for ket in kets]
    sites = np.sort(sites)

    d = kets[0].shape[1]
    k_tot = len(sites)
    mid_up = [1]
    mid_down = [2]

    # left env:
    env_l = np.identity(n=mps[sites[0]-1].shape[0])
    label_env = mid_up + mid_down
    if prnt:
        print(label_env, env_l.shape)
    # right env:
    env_r = np.identity(n=mps[sites[-1]-1].shape[-1])
    
    # central env
    i = 0
    for j in range(sites[0],sites[-1]+1):
        if j in sites:
            if prnt:
                print("j in sites: ", j)
            label_ket = [1, -1 - i, -2*k_tot -1]
            label_bra = [2, -k_tot - 1 - i, -2*k_tot - 2]
            if prnt:
                print(label_ket, label_bra)
            env_l = ncon(
                [env_l, kets[j - 1], bras[j - 1]],
                [label_env, label_ket, label_bra],
            )
            up = [int(-elem) for elem in np.linspace(1, i + 1, i + 1)]
            down = [int(-elem) for elem in np.linspace(k_tot + 1, k_tot + 1 + i, i + 1)]
            label_env = up + down + mid_up + mid_down
            if prnt:
                print(label_env, env_l.shape)

            i += 1
        else:
            if prnt:
                print("j out sites: ", j)
            label_ket = [1, 3, -2*k_tot-1]
            label_bra = [2, 3, -2*k_tot-2]
            if prnt:
                print(label_ket, label_bra)
            env_l = ncon([env_l, kets[j-1], bras[j-1]], [label_env, label_ket, label_bra])
        
    rdm = ncon([env_l, env_r], [label_env, [1, 2]]).reshape((d**k_tot, d**k_tot))

    return rdm

In [8]:
sites = [3,4,5,8,9]
psi_mcf = left_canonical_form(mps=psi_rcf, stop=sites[0])
rdm = generalized_k_rdm(mps=psi_mcf, sites=sites, prnt=True)
rdm.shape

[1, 2] (4, 4)
j in sites:  3
[1, -1, -11] [2, -6, -12]
[-1, -6, 1, 2] (2, 2, 8, 8)
j in sites:  4
[1, -2, -11] [2, -7, -12]
[-1, -2, -6, -7, 1, 2] (2, 2, 2, 2, 16, 16)
j in sites:  5
[1, -3, -11] [2, -8, -12]
[-1, -2, -3, -6, -7, -8, 1, 2] (2, 2, 2, 2, 2, 2, 32, 32)
j out sites:  6
[1, 3, -11] [2, 3, -12]
j out sites:  7
[1, 3, -11] [2, 3, -12]
j in sites:  8
[1, -4, -11] [2, -9, -12]
[-1, -2, -3, -4, -6, -7, -8, -9, 1, 2] (2, 2, 2, 2, 2, 2, 2, 2, 64, 64)
j in sites:  9
[1, -5, -11] [2, -10, -12]
[-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, 1, 2] (2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 64, 64)


(32, 32)

In [12]:
sites=[10]
print(f"time for left canonical form")
%timeit left_canonical_form(mps=psi_rcf, stop=sites[0])
psi_mcf = left_canonical_form(mps=psi_rcf, stop=sites[0])
for site in range(11,17):
    print(f"time for {len(sites)} site(s) rmd")
    %timeit generalized_k_rdm(mps=psi_mcf, sites=sites)
    sites = sites + [site]

time for left canonical form
18.5 ms ± 914 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)
time for 1 site(s) rmd
1.25 ms ± 75.6 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
time for 2 site(s) rmd
2.14 ms ± 56.6 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)
time for 3 site(s) rmd
4.85 ms ± 266 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)
time for 4 site(s) rmd
16.7 ms ± 38.7 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)
time for 5 site(s) rmd
58 ms ± 461 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)
time for 6 site(s) rmd
357 ms ± 2.4 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


## Check the properties of the Density Matrix

In [137]:
from scipy.sparse import csr_matrix
from qs_mps.utils import truncation
from qs_mps.checks import check_matrix, check_hermitian

def _compute_norm(mps, site, prnt=False):
    """
    _compute_norm

    This function computes the norm of our quantum state which is represented in mps.
    It takes the attributes .sites and .bonds of the class which gives us
    mps in canonical form (Vidal notation).

    site: int - sites in the chain
    ancilla: bool - if True we compute the norm of the ancilla_sites mps. By default False
    mixed: bool - if True we compute the braket between the ancilla_sites and the sites mps. By default False

    """
    a = np.array([1])

    array = mps.copy()

    ten = ncon([a, a],[[-1], [-2]])
    for i in range(site):
        
        ten = ncon(
            [ten, array[i], array[i].conjugate()],
            [[1, 2], [1, 3, -1], [2, 3, -2]],
        )
        if prnt:
            print(f"site {i+1}: ", ten.shape, array[i].shape)

    N = ncon([ten, a, a], [[1, 2], [1], [2]]).real

    return N

def check_canonical(mps, site: int, prnt: bool=False):
    """
    check_canonical

    This funciton checks if the tensor at a certain site is in the correct
    mixed canonical form, e.g., LCF on the left of the site and RCF on the right of the site.

    site: int - where we want to observe the canonical form of our mps

    """
    assert site < len(mps) and site > 0, f"The site chosen is out of range (0,{len(mps)})"
    array = mps.copy()
    a = np.array([1])

    env = ncon([a, a], [[-1], [-2]])
    env_l = env
    all_good = 0
    for i in range(0, site - 1):
        I = np.eye(array[i].shape[2])

        if prnt:
            print(f"site {i+1}: ", env_l.shape, array[i].shape)
        env_l = ncon(
            [env_l, array[i], (array[i].conjugate())],
            [[1, 2], [1, 3, -1], [2, 3, -2]],
        )
        env_trunc = truncation(env_l, threshold=1e-15)
        env_trunc = csr_matrix(env_trunc)
        I = csr_matrix(I)
        ratio = check_matrix(env_trunc, I)
        if ratio < 1e-12:
            if prnt:
                print(f"the tensor at site {i+1} is in the correct LCF")
                all_good += 1
        else:
            if prnt:
                print(f"the  tensor at site {i+1} has a ratio with the identity matrix equal to: {ratio}")

    env_r = env
    for i in range(len(mps) - 1, site, -1):
        if prnt:
            print(f"site {i+1}: ", env_r.shape, array[i].shape)
        I = np.eye(array[i].shape[0])
        env_r = ncon(
            [array[i], (array[i].conjugate()), env_r],
            [[-1, 3, 1], [-2, 3, 2], [1, 2]],
        )
        env_trunc = truncation(env_r, threshold=1e-15)
        env_trunc = csr_matrix(env_trunc)
        I = csr_matrix(I)
        ratio = check_matrix(env_trunc, I)
        if ratio < 1e-12:
            if prnt:
                print(f"the tensor at site {i+1} is in the correct RCF")
                all_good += 1
        else:
            if prnt:
                print(f"the  tensor at site {i+1} has a ratio with the identity matrix equal to: {ratio}")
    return all_good

In [138]:
sites = [27,30,35]
# rdm = ac_rdm(mps=psi_rcf, sites=sites)

psi_mcf = left_canonical_form(mps=psi_rcf, stop=sites[0])
rdm = generalized_k_rdm(mps=psi_mcf, sites=sites, prnt=False)

# trace of the rdm
print("trace: ", rdm.trace())

# hermiticity of the rdm
ratio = check_hermitian(csr_matrix(rdm))

trace:  (64.00000000000007+0j)
CHECK HERMITICITY
The matrix is hermitian


In [157]:
sites = [10,11,12]
norm = _compute_norm(psi_rcf, len(psi_rcf), prnt=False)
print(f"norm before mixed c.f.: {norm}")
psi_mcf = left_canonical_form(mps=psi_rcf, stop=sites[0])
norm = _compute_norm(psi_mcf, len(psi_mcf), prnt=False)
print(f"norm after mixed c.f.: {norm}")
psi_mcf[sites[0]] = psi_mcf[sites[0]]/(norm**(1/2))
norm = _compute_norm(psi_mcf, len(psi_mcf), prnt=False)
print(f"norm after mixed c.f.: {norm}")
check_canonical(psi_mcf, sites[0], prnt=True)
# _compute_norm(psi_mcf_new, 50, prnt=True)

norm before mixed c.f.: 1.0000000000000029
norm after mixed c.f.: 64.00000000000003
norm after mixed c.f.: 1.0000000000000004
site 1:  (1, 1) (1, 2, 2)
the tensor at site 1 is in the correct LCF
site 2:  (2, 2) (2, 2, 4)
the tensor at site 2 is in the correct LCF
site 3:  (4, 4) (4, 2, 8)
the tensor at site 3 is in the correct LCF
site 4:  (8, 8) (8, 2, 16)
the tensor at site 4 is in the correct LCF
site 5:  (16, 16) (16, 2, 32)
the tensor at site 5 is in the correct LCF
site 6:  (32, 32) (32, 2, 55)
the tensor at site 6 is in the correct LCF
site 7:  (55, 55) (55, 2, 54)
the tensor at site 7 is in the correct LCF
site 8:  (54, 54) (54, 2, 62)
the tensor at site 8 is in the correct LCF
site 9:  (62, 62) (62, 2, 64)
the tensor at site 9 is in the correct LCF
site 50:  (1, 1) (2, 2, 1)
the tensor at site 50 is in the correct RCF
site 49:  (2, 2) (4, 2, 2)
the tensor at site 49 is in the correct RCF
site 48:  (4, 4) (8, 2, 4)
the tensor at site 48 is in the correct RCF
site 47:  (8, 8) (1

48

In [169]:
sites = [10,11,12,13]

psi_mcf = left_canonical_form(mps=psi_rcf, stop=sites[0])
norm = _compute_norm(psi_mcf, len(psi_mcf), prnt=False)
psi_mcf[sites[0]] = psi_mcf[sites[0]]/(norm**(1/2))
rdm = generalized_k_rdm(mps=psi_mcf, sites=sites, prnt=False)

# trace of the rdm
print("trace: ", rdm.trace())

# hermiticity of the rdm
ratio = check_hermitian(csr_matrix(rdm))

trace:  (0.9999999999999989+0j)
CHECK HERMITICITY
The matrix is hermitian


## Check the two matrices are the same

In [2]:
filename = "results/data/ANNNI-3f848197-11cd-4ce7-8e20-4166478a5ac9.pkl" # ANNNI 10, c1=-1, upside down, 10 x 10, all phase diagram

with gzip.open(filename, 'rb') as f:
    data = pickle.load(f)
params = data['params']
l, n = data['l'], data['n']
gstates = data['gstates']
stats = data['stats']

params_extent = np.concatenate([np.min(params, axis=0), np.max(params, axis=0)])
params_extent = tuple(params_extent[[0, 2, 1, 3]])

In [5]:
from qphaset.phases import gstates_to_rdms_matrix_qs_mps
sites = [l // 2, (l // 2) + 1]
# sites = [l // 2]

rdms_old = gstates_to_rdms_matrix_qs_mps(gstates, sites=sites, generalized=False)
rdms_new = gstates_to_rdms_matrix_qs_mps(gstates, sites=sites, generalized=True)

In [6]:
[np.isclose(rdms_old[i],rdms_new[i]) for i in range(len(rdms_new))]

[array([[[ True,  True,  True,  True],
         [ True,  True,  True,  True],
         [ True,  True,  True,  True],
         [ True,  True,  True,  True]],
 
        [[ True,  True,  True,  True],
         [ True,  True,  True,  True],
         [ True,  True,  True,  True],
         [ True,  True,  True,  True]],
 
        [[ True,  True,  True,  True],
         [ True,  True,  True,  True],
         [ True,  True,  True,  True],
         [ True,  True,  True,  True]],
 
        [[ True,  True,  True,  True],
         [ True,  True,  True,  True],
         [ True,  True,  True,  True],
         [ True,  True,  True,  True]],
 
        [[ True,  True,  True,  True],
         [ True,  True,  True,  True],
         [ True,  True,  True,  True],
         [ True,  True,  True,  True]],
 
        [[ True,  True,  True,  True],
         [ True,  True,  True,  True],
         [ True,  True,  True,  True],
         [ True,  True,  True,  True]],
 
        [[ True,  True,  True,  True],
       

In [7]:
rdms_old[0,0], rdms_new[0,0]

(array([[ 0.9230106 +0.j, -0.02011479+0.j, -0.05264315+0.j,
          0.14850038+0.j],
        [-0.02011479+0.j,  0.02584496+0.j,  0.0115613 +0.j,
         -0.00448401+0.j],
        [-0.05264315+0.j,  0.0115613 +0.j,  0.0266689 +0.j,
         -0.00885095+0.j],
        [ 0.14850038+0.j, -0.00448401+0.j, -0.00885095+0.j,
          0.02447554+0.j]]),
 array([[ 0.9230106 +0.j, -0.02011479+0.j, -0.05264315+0.j,
          0.14850038+0.j],
        [-0.02011479+0.j,  0.02584496+0.j,  0.0115613 +0.j,
         -0.00448401+0.j],
        [-0.05264315+0.j,  0.0115613 +0.j,  0.0266689 +0.j,
         -0.00885095+0.j],
        [ 0.14850038+0.j, -0.00448401+0.j, -0.00885095+0.j,
          0.02447554+0.j]]))